# Lo que el modelo no puede saber

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

| | |
|---|---|
| **Problema** | Un modelo responde con lo que vio al entrenarse. No conoce los horarios de una biblioteca barrial ni la fecha de hoy, y aun así contesta. |
| **Producto** | Una función que responde con un documento, una medición de cuándo el documento deja de alcanzar y una regla para elegir entre cuatro estrategias. |
| **Recursos** | Ollama con `granite4:latest`. La sección 3 hace cuatro llamadas largas (unos 30 segundos cada una). La búsqueda web (SerpAPI) y Gemini son opcionales. |

Una vecina pregunta hasta qué hora abre la biblioteca popular del barrio los sábados. El equipo municipal ya tiene el reglamento escrito. El modelo, en cambio, no lo leyó nunca: lo que sabe sobre esa biblioteca es lo que se pueda deducir de bibliotecas en general.

**¿Qué hacemos cuando la respuesta depende de un dato que el modelo nunca vio, y cuándo ese dato deja de entrar en el pedido?**

> **Cómo se trabaja.** En clase se presenta cada tema con el código ya escrito. Después lo explorás por tu cuenta: ejecutás las celdas, leés las notas **Para mirar** y cambiás valores para ver qué pasa. Al final de cada bloque hay un **microlaboratorio**: una tabla te dice qué líneas completar y con qué, y la celda siguiente comprueba si quedó bien.

## Los datos

| Campo | Dato |
|---|---|
| Fuente | `politicas_biblioteca.md` de la unidad 010: reglamento sintético de la «Biblioteca Popular Almafuerte», CC0-1.0 |
| Unidad de análisis | Un documento breve (cuatro secciones) y preguntas sobre él |
| Texto de relleno | Para la sección 3 se genera un texto largo con avisos internos **inventados** que no tienen relación con las preguntas. Sirve para provocar una falla, no para describir ninguna biblioteca real. |
| Búsqueda web | Sin clave, la sección 4 usa cuatro resultados **sintéticos** con la forma de una respuesta de SerpAPI. No son resultados reales. |
| Limitaciones | Un solo documento y un solo modelo. Lo que se observa describe a este modelo con este texto, no a «los modelos». |

In [1]:
import datetime
import json
import os

import utils

utils.comprobar_modelo_local()
utils.versiones(["ollama", "google-search-results", "pandas"])

Ollama responde en http://localhost:11434 y tiene granite4:latest.


{'python': '3.12.13',
 'ollama': '0.6.3',
 'google-search-results': '2.4.2',
 'pandas': '3.1.0'}

## 1. Lo que sabe y lo que inventa

Preguntamos tres cosas: un dato general y estable, un dato privado de una biblioteca barrial y la fecha de hoy.

In [2]:
preguntas_sin_documento = [
    "¿Cuál es la capital de la provincia de Santa Fe? Respondé en una "
    "palabra.",
    "¿Hasta qué hora abre la Biblioteca Popular Almafuerte los sábados?",
    "¿Qué día es hoy? Respondé con la fecha.",
]

filas_sin_documento = []
for pregunta in preguntas_sin_documento:
    mensajes = [{"role": "user", "content": pregunta}]
    resultado = utils.preguntar(mensajes, maximo_tokens=60)
    filas_sin_documento.append({
        "pregunta": pregunta,
        "respuesta": resultado["texto"],
    })

print("Fecha real:", datetime.date.today().isoformat())
utils.tabla(filas_sin_documento)

Fecha real: 2026-10-23


,pregunta,respuesta
0,¿Cuál es la capital de la provincia de Santa Fe? Respondé en una palabra.,Santa Fe.
1,¿Hasta qué hora abre la Biblioteca Popular Almafuerte los sábados?,"La Biblioteca Popular de Almafuerte, en general, suele tener horarios de apertura que pueden variar dependiendo del día de la semana. Sin embargo, los fines de semana, como el sábado, las bibliotecas suelen estar abiertas con horarios más amplios"
2,¿Qué día es hoy? Respondé con la fecha.,El 15 de abril de 2025.


> **Para mirar.**
>
> - La capital la sabe: es un dato general y estable.
> - Sobre la biblioteca no dio un horario, pero respondió con generalidades que suenan a respuesta.
> - A «¿qué día es hoy?» contestó una fecha concreta que no coincide con la fecha real impresa arriba. El modelo no tiene reloj y no lo dice.

## 2. Darle el texto

La solución más simple es pegar el documento en el pedido y ordenar: «respondé solo con esto; si no figura, decilo». El modelo no *aprende* el reglamento: lo lee de nuevo en cada llamada.

In [3]:
documento = utils.cargar_texto("politicas_biblioteca.md")
print(documento)

# Políticas de la Biblioteca Popular Almafuerte

## Horarios de atención

La biblioteca atiende de lunes a viernes de 9 a 19 horas. Los sábados abre de 10 a 14 horas. Los domingos y feriados permanece cerrada.

## Préstamo de libros

Cada socio puede llevarse hasta tres libros por préstamo. El préstamo dura 14 días corridos y se puede renovar una vez, de manera presencial o por teléfono, si nadie reservó el libro.

## Devolución

Los libros se devuelven en el mostrador de lunes a sábado, en el horario de atención. Deben volver en el mismo estado en que se prestaron, sin hojas sueltas ni anotaciones. Los libros de consulta de la sala de lectura no se prestan a domicilio.

## Actividades culturales

Los martes a las 18 horas hay un taller de escritura abierto. El primer sábado de cada mes se realiza una feria de intercambio de libros. Las actividades son gratuitas y no requieren inscripción previa.


In [4]:
regla = utils.REGLA_DE_DOCUMENTO
pregunta = "¿Hasta qué hora abre la biblioteca los sábados?"

mensajes = [
    {"role": "system", "content": regla},
    {
        "role": "user",
        "content": f"Documento:\n{documento}\n\nPregunta: {pregunta}",
    },
]
resultado = utils.preguntar(mensajes)
print(resultado["texto"])
print(f"Tokens del pedido: {resultado['tokens_entrada']}")

De 10 a 14 horas.
Tokens del pedido: 320


Probamos cinco preguntas y comparamos con la respuesta esperada. Una es **trampa**: el reglamento no dice cuánto cuesta asociarse, así que lo correcto es reconocerlo.

In [5]:
preguntas_con_respuesta = [
    {
        "pregunta": "¿Hasta qué hora abre la biblioteca los sábados?",
        "debe_incluir": "14",
    },
    {
        "pregunta": "¿Cuántos libros puedo llevarme por préstamo?",
        "debe_incluir": "tres",
    },
    {
        "pregunta": "¿Se puede renovar un préstamo por teléfono?",
        "debe_incluir": "teléfono",
    },
    {
        "pregunta": "¿Qué día hay taller de escritura?",
        "debe_incluir": "martes",
    },
    {"pregunta": "¿Cuánto cuesta asociarse?", "debe_incluir": "no figura"},
]

filas_con_documento = []
for caso in preguntas_con_respuesta:
    resultado = utils.responder_con_documento(documento, caso["pregunta"])
    texto = resultado["texto"]
    esta_bien = caso["debe_incluir"] in texto.lower()
    filas_con_documento.append({
        "pregunta": caso["pregunta"],
        "respuesta": texto,
        "esperado": caso["debe_incluir"],
        "coincide": esta_bien,
    })

utils.tabla(filas_con_documento)

,pregunta,respuesta,esperado,coincide
0,¿Hasta qué hora abre la biblioteca los sábados?,De 10 a 14 horas.,14,True
1,¿Cuántos libros puedo llevarme por préstamo?,Tres libros.,tres,True
2,¿Se puede renovar un préstamo por teléfono?,"Sí, se puede renovar un préstamo por teléfono.",teléfono,True
3,¿Qué día hay taller de escritura?,Los martes a las 18 horas hay un taller de escritura abierto.,martes,True
4,¿Cuánto cuesta asociarse?,No figura en el documento.,no figura,True


> **Para mirar.**
>
> - Las cinco respuestas coinciden con lo esperado, incluida la pregunta trampa: respondió «No figura en el documento» en lugar de rellenar con generalidades, como en la sección 1.
> - Son cinco preguntas sobre un texto de unos 320 tokens. Muestra que el mecanismo funciona; no mide qué tan seguido acierta.

### Microlaboratorio 1 — La función que responde con un documento

**Tarea.** Encapsular los pasos de la sección 2 en una función.  
**Producto.** `responder_con_documento`, comprobada con dos preguntas.  
**Antes de empezar.** Sección 2.

| Línea | Qué tiene que quedar | Con qué |
|---|---|---|
| (1) `contenido` | el documento seguido de la pregunta | un f-string con `Documento:`, `documento`, `Pregunta:` y `pregunta`, como en la sección 2 |
| (2) `texto` | solo el texto de la respuesta | la clave `"texto"` del diccionario que devuelve `utils.preguntar(mensajes)` |

**Comprobación.** Una pregunta con respuesta en el documento y una que no figura.

> **Ayuda 1.** El `content` del mensaje `user` de la sección 2 ya tiene la forma que necesitás.  
> **Ayuda 2.** `utils.preguntar` devuelve un diccionario; el texto está en la clave `"texto"`.

In [6]:
def responder_con_documento(documento: str, pregunta: str) -> str:
    """Responde una pregunta usando solo el texto de un documento.

    Args:
        documento: Texto que se pega en el pedido.
        pregunta: Pregunta de la persona.

    Returns:
        El texto de la respuesta del modelo.
    """
    contenido = pregunta  # COMPLETAR (1)
    mensajes = [
        {"role": "system", "content": utils.REGLA_DE_DOCUMENTO},
        {"role": "user", "content": contenido},
    ]
    texto = ""  # COMPLETAR (2)
    return texto


con_respuesta = responder_con_documento(
    documento, "¿Hasta qué hora abre la biblioteca los sábados?"
)
sin_respuesta = responder_con_documento(documento, "¿Cuánto cuesta asociarse?")
utils.comprobar(
    "14" in con_respuesta,
    "revisa (1) y (2): la respuesta tiene que mencionar las 14 horas",
)
utils.comprobar(
    "no figura" in sin_respuesta.lower(),
    "revisa (1) y (2): la pregunta trampa tiene que dar No figura",
)

Todavia no: revisa (1) y (2): la respuesta tiene que mencionar las 14 horas
Todavia no: revisa (1) y (2): la pregunta trampa tiene que dar No figura


A partir de acá usamos la versión completa, `utils.responder_con_documento`, que además deja controlar el tamaño del contexto.

## 3. Cuando el documento no entra

Un modelo lee una cantidad máxima de tokens por llamada: su **ventana de contexto**. En Ollama el valor por defecto es `num_ctx`, y si el pedido es más largo, **recorta el principio sin avisar**. Armamos un texto de unos 5500 tokens: el reglamento más 120 avisos internos de relleno. Hacemos la misma pregunta con el reglamento al principio y al final, con la ventana por defecto y con una de 16384 tokens.

In [7]:
dias = ["lunes", "martes", "miércoles", "jueves", "viernes"]

avisos_de_relleno = []
for numero in range(1, 121):
    dia = dias[numero % 5]
    avisos_de_relleno.append(
        f"Aviso interno {numero}: la sala {numero % 9 + 1} del segundo "
        f"piso se reserva los {dia} para grupos de estudio de "
        f"{numero % 7 + 3} personas; el responsable de turno es el "
        f"legajo {1000 + numero}."
    )
relleno = "\n".join(avisos_de_relleno)

textos_largos = {
    "reglamento al principio": documento + "\n\n" + relleno,
    "reglamento al final": relleno + "\n\n" + documento,
}
texto_de_control = textos_largos["reglamento al final"]
print(f"Palabras del texto largo: {len(texto_de_control.split())}")

Palabras del texto largo: 3518


In [8]:
pregunta = "¿Hasta qué hora abre la biblioteca los sábados?"

filas_de_contexto = []
for lugar, texto_largo in textos_largos.items():
    for contexto in (None, 16384):
        resultado = utils.responder_con_documento(
            texto_largo, pregunta, contexto=contexto, maximo_tokens=60
        )
        filas_de_contexto.append({
            "dónde está el reglamento": lugar,
            "ventana (num_ctx)": contexto or "por defecto",
            "tokens leídos": resultado["tokens_entrada"],
            "segundos": round(resultado["segundos"], 1),
            "respuesta": resultado["texto"],
        })

utils.tabla(filas_de_contexto)

,dónde está el reglamento,ventana (num_ctx),tokens leídos,segundos,respuesta
0,reglamento al principio,por defecto,4096,21.7,"La información proporcionada no incluye detalles específicos sobre las horas de apertura de la biblioteca los sábados. Para obtener esa información, se recomienda contactar directamente a la biblioteca o visitar su sitio web oficial, donde generalmente se detallan las horas de funcion"
1,reglamento al principio,16384,5264,25.5,No figura en el documento.
2,reglamento al final,por defecto,4096,19.0,La biblioteca Popular Almafuerte abre los sábados de 10 a 14 horas.
3,reglamento al final,16384,5264,25.2,10 a 14 horas.


> **Para mirar.**
>
> - Con la ventana por defecto, Ollama leyó 4096 tokens aunque el pedido tenía 5264: **recortó el principio sin avisar**. Con el reglamento al principio, el modelo contestó con una evasiva; con el reglamento al final, encontró «10 a 14 horas».
> - Con 16384 tokens entró todo, pero con el reglamento al principio la respuesta fue «No figura en el documento». El dato **entraba y el modelo no lo usó**.
> - Cada llamada tardó decenas de segundos, contra una fracción de segundo con el reglamento solo.
> - Es una prueba con un modelo y un texto: no alcanza para afirmar cómo se comportan «los modelos» con datos al principio.

In [9]:
if utils.hay_clave("GEMINI_API_KEY"):
    texto_para_gemini = textos_largos["reglamento al principio"]
    consulta = f"Documento:\n{texto_para_gemini}\n\nPregunta: {pregunta}"
    resultado_remoto = utils.preguntar_a_gemini(
        utils.REGLA_DE_DOCUMENTO, consulta
    )
    print(resultado_remoto["texto"])
    print(f"Tokens del pedido: {resultado_remoto['tokens_entrada']}")
else:
    utils.avisar_sin_clave("GEMINI_API_KEY")

Sin GEMINI_API_KEY: se omite esta parte. Copia .env.example como .env, completa la clave y reinicia el kernel.


> **Para mirar.** Con clave, compará con la fila «reglamento al principio» y ventana de 16384. Que un modelo remoto lea el dato describe a ese modelo con este texto; no prueba que un modelo más grande «siempre» encuentre datos en textos largos.

## 4. Buscar afuera

Cuando el dato no está en ningún documento propio (una jornada de vacunación de esta semana, por ejemplo), una opción es **buscar en la web** y pegar los resultados como contexto. El patrón es el mismo de la sección 2, con un paso previo: buscar. Usamos **SerpAPI**, que devuelve los resultados de Google como datos. Tiene un plan gratuito con pocas búsquedas por mes: la clave va en `.env`, nunca en el cuaderno. Sin clave, usamos cuatro resultados **sintéticos** del archivo `busqueda_ejemplo.json`.

In [10]:
consulta_web = "jornada de vacunación antigripal en Rosario esta semana"


def buscar_en_la_web(consulta: str, cantidad: int = 4) -> list[dict]:
    """Busca en Google con SerpAPI, o usa resultados de ejemplo.

    Args:
        consulta: Texto a buscar.
        cantidad: Cuantos resultados se piden.

    Returns:
        Lista de resultados con "title", "link" y "snippet".
    """
    if not utils.hay_clave("SERPAPI_API_KEY"):
        utils.avisar_sin_clave("SERPAPI_API_KEY")
        print("Se usan resultados SINTETICOS de datos/busqueda_ejemplo.json")
        muestra = utils.cargar_json("busqueda_ejemplo.json")
        return muestra["organic_results"][:cantidad]
    from serpapi import GoogleSearch

    busqueda = GoogleSearch({
        "q": consulta,
        "api_key": os.getenv("SERPAPI_API_KEY"),
        "num": cantidad,
        "hl": "es",
        "gl": "ar",
    })
    return busqueda.get_dict().get("organic_results", [])[:cantidad]


resultados_web = buscar_en_la_web(consulta_web)
filas_de_busqueda = []
numero = 0
for resultado in resultados_web:
    numero += 1
    filas_de_busqueda.append({
        "n": numero,
        "título": resultado["title"],
        "fragmento": resultado["snippet"],
    })
utils.tabla(filas_de_busqueda)

Sin SERPAPI_API_KEY: se omite esta parte. Copia .env.example como .env, completa la clave y reinicia el kernel.
Se usan resultados SINTETICOS de datos/busqueda_ejemplo.json


,n,título,fragmento
0,1,Vacunación antigripal: cronograma de la semana,"El cronograma de la semana incluye jornadas de vacunación antigripal en centros de salud de los seis distritos, de lunes a viernes de 8 a 14 horas, con DNI."
1,2,Campaña 2019: vacunación antigripal gratuita,En 2019 la campaña de vacunación antigripal se realizó en escuelas y plazas durante el mes de abril.
2,3,Qué es la gripe y cómo se previene,"La gripe es una infección respiratoria. La vacunación anual, el lavado de manos y la ventilación de ambientes reducen el contagio."
3,4,Horarios del transporte urbano,Consulte los recorridos y frecuencias de las líneas de colectivo durante el fin de semana.


Convertimos los resultados en un contexto numerado y le pedimos al modelo que **cite el número** de cada afirmación, para que quien lee pueda volver a la fuente. Probamos dos reglas y dos preguntas: una que los resultados responden y otra que no.

In [11]:
contexto_web = ""
numero = 0
for resultado in resultados_web:
    numero += 1
    contexto_web += (
        f"[{numero}] {resultado['title']}. {resultado['snippet']} "
        f"({resultado['link']})\n"
    )

reglas_web = {
    "estricta": (
        "Respondé usando solo los resultados de búsqueda. Citá el número "
        "entre corchetes después de cada dato. Si los resultados no "
        "alcanzan, respondé exactamente: Los resultados no alcanzan."
    ),
    "con criterio": (
        "Respondé usando solo los resultados de búsqueda. Si algún "
        "resultado contiene la respuesta, respondé y citá el número entre "
        "corchetes después de cada dato. Solo si ninguno la contiene, "
        "respondé exactamente: Los resultados no alcanzan. Ignorá los "
        "resultados que no tengan relación con la pregunta o que sean de "
        "años anteriores."
    ),
}
preguntas_web = [
    "¿Cuándo hay jornadas de vacunación antigripal en Rosario esta semana?",
    "¿Cuánto cuesta la vacuna antigripal?",
]

filas_web = []
for nombre_de_regla, regla_web in reglas_web.items():
    for pregunta_web in preguntas_web:
        mensajes_web = [
            {"role": "system", "content": regla_web},
            {
                "role": "user",
                "content": (
                    f"Resultados:\n{contexto_web}\n"
                    f"Pregunta: {pregunta_web}"
                ),
            },
        ]
        respuesta_web = utils.preguntar(mensajes_web)
        filas_web.append({
            "regla": nombre_de_regla,
            "pregunta": pregunta_web,
            "respuesta": respuesta_web["texto"],
        })

utils.tabla(filas_web)

,regla,pregunta,respuesta
0,estricta,¿Cuándo hay jornadas de vacunación antigripal en Rosario esta semana?,Los resultados no alcanzan.
1,estricta,¿Cuánto cuesta la vacuna antigripal?,Los resultados no alcanzan.
2,con criterio,¿Cuándo hay jornadas de vacunación antigripal en Rosario esta semana?,"[1] Vacunación antigripal: cronograma de la semana. El cronograma de la semana incluye jornadas de vacunación antigripal en centros de salud de los seis distritos, de lunes a viernes de 8 a 14 horas, con DNI."
3,con criterio,¿Cuánto cuesta la vacuna antigripal?,Los resultados no alcanzan.


> **Para mirar.**
>
> - Con los resultados sintéticos de ejemplo, la regla `estricta` respondió «Los resultados no alcanzan» a las dos preguntas aunque el resultado 1 contenía la respuesta a la primera. La regla `con criterio` respondió la primera citando `[1]` y rechazó la segunda, que ningún resultado contesta.
> - El resultado de 2019 y los que no tratan de vacunación estaban ahí para confundir: una buena regla los ignora, pero eso hay que verificarlo cada vez.
> - Si usaste SerpAPI, tus resultados son otros: revisá que cada dato tenga su número y que la fuente sea reciente.

### Microlaboratorio 2 — Elegir la estrategia

**Tarea.** Asignar a cada pregunta la estrategia que corresponde, según lo que mediste.  
**Producto.** `estrategias` y `mi_decision`.  
**Antes de empezar.** Secciones 1 a 4.

Las estrategias posibles son `"modelo solo"`, `"pegar el texto"`, `"recuperar fragmentos"` y `"buscar en la web"`. La tercera todavía no la construimos: es la que viene en la unidad 014 y consiste en seleccionar, de un conjunto grande de documentos, solo los fragmentos que importan para cada pregunta.

| Pregunta | Qué tiene que quedar |
|---|---|
| (1) la capital de Santa Fe | una estrategia, según la sección 1 |
| (2) el horario de la biblioteca | una estrategia, según la sección 2 |
| (3) un manual de 300 páginas | una estrategia, según la sección 3 |
| (4) la vacunación de esta semana | una estrategia, según la sección 4 |

En `mi_decision`, explicá en al menos 30 palabras **por qué una de tus elecciones tiene un riesgo que las otras no tienen**.

> **Ayuda 1.** La sección 3 muestra que agrandar la ventana no alcanzó: el modelo no encontró el dato aunque entraba.  
> **Ayuda 2.** En la sección 4, mirá si los resultados de búsqueda eran todos pertinentes y recientes.

In [12]:
estrategias = {
    "la capital de Santa Fe": "modelo solo",
    "el horario de la biblioteca": "modelo solo",  # COMPLETAR (2)
    "un manual de 300 páginas": "modelo solo",  # COMPLETAR (3)
    "la vacunación de esta semana": "modelo solo",  # COMPLETAR (4)
}

opciones = [
    "modelo solo", "pegar el texto", "recuperar fragmentos", "buscar en la web"
]
todas_validas = True
for pregunta_elegida, estrategia in estrategias.items():
    if estrategia not in opciones:
        todas_validas = False
utils.comprobar(todas_validas, "usa solo las cuatro estrategias del enunciado")
utils.comprobar(
    estrategias["el horario de la biblioteca"] == "pegar el texto",
    "revisa (2): el reglamento es corto, entra y se puede pegar",
)
utils.comprobar(
    estrategias["un manual de 300 páginas"] == "recuperar fragmentos",
    "revisa (3): un manual largo no entra, o entra y se pierde el dato",
)
utils.comprobar(
    estrategias["la vacunación de esta semana"]
    == "buscar en la web",
    "revisa (4): el dato es reciente y no esta en un documento propio",
)

mi_decision = ""
utils.comprobar(
    len(mi_decision.split()) >= 30,
    "escribi al menos 30 palabras en mi_decision",
)

Correcto.
Todavia no: revisa (2): el reglamento es corto, entra y se puede pegar
Todavia no: revisa (3): un manual largo no entra, o entra y se pierde el dato
Todavia no: revisa (4): el dato es reciente y no esta en un documento propio
Todavia no: escribi al menos 30 palabras en mi_decision


## Qué te llevás

- Un modelo **no sabe lo que no vio** y, si no se lo decimos, contesta igual: sobre la biblioteca dio una respuesta vaga y sobre la fecha de hoy, una fecha inventada.
- Pegar el documento resuelve los casos cortos, pero **el pedido tiene un tope** que Ollama aplica sin avisar, y que el dato entre en la ventana **no garantiza que el modelo lo use**.
- Buscar afuera agrega otra fuente de error: resultados viejos o ajenos a la pregunta. La respuesta tiene que **citar** de dónde sale lo que dice.

**Sigue:** la unidad 014 (RAG), que selecciona por pregunta solo los fragmentos pertinentes de una colección de documentos en lugar de pegarlos todos.